# Notebook 57 — Numeric Feature Engineering

**Hypothesis**: The 10 raw numeric features (SNIP, CiteScore, SJR, Topic Prominence,
author/institution/country counts) contain latent interactions that a linear scaler
cannot expose. Adding hand-crafted interaction terms and ratios will improve F1/AUC
on top of the best SPECTER1 baseline (nb55-C).

**Prior art**:
- `nb55-C`: SPECTER1 + 10 numerics, LGBM tuned, merged→AUB — best result so far.
- `nb55-E`: numeric-only, LGBM, merged→AUB — shows numerics carry real signal alone.
- REF-CLEAN: F1=0.5109, AUC=0.8216.

**Approach**:
- Build an **extended numeric block** (10 raw + interaction/ratio features)
- Ablation A: extended numerics only (LGBM tuned) — quantifies pure numeric gain
- Ablation B: SPECTER1 + extended numerics (LGBM tuned) — main experiment
- Compare directly against nb55-C (SPECTER1 + raw numerics, LGBM tuned)

**New features**:

| Feature | Formula | Rationale |
|---------|---------|----------|
| `venue_quality` | `citescore_pct × snip_pct` | Combined venue prestige signal |
| `venue_momentum` | `citescore_pct × topic_prom` | Prestige in a growing field |
| `venue_disagreement` | `\|sjr_pct − snip_pct\|` | Metric disagreement → niche journals |
| `collab_breadth` | `num_countries / num_authors` | International reach per author |
| `inst_diversity` | `num_institutions / num_authors` | Institutional spread per author |
| `multi_country` | `num_countries > 1` (binary) | Any international collaboration |
| `author_load` | `num_authors / num_institutions` | Authors per institution |
| `top_venue_topic` | `snip × topic_prom / 100` | Raw prestige × topic prominence |


In [ ]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
import copy
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, cohen_kappa_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
TRAIN_YEARS  = list(range(2010, 2018))
TEST_YEARS   = [2018, 2019, 2020]
QUANTILE     = 0.75

# Reference values — fill in nb55-C result once known
NB54_REF_CLEAN_F1  = 0.5109
NB54_REF_CLEAN_AUC = 0.8216
NB54_BEST_F1       = 0.5302
NB54_BEST_AUC      = 0.8257
NB55_C_F1          = None   # TODO: fill in from nb55
NB55_C_AUC         = None

CACHE_DIR = Path('../../data/cache')
print('Libraries loaded')

## 1. Load data & splits

In [ ]:
data_path = Path('../../data/processed/all_unis_cleaned.pkl')
df = pd.read_pickle(data_path)

df_train     = df[df['Year'].isin(TRAIN_YEARS)].copy()
df_test_all  = df[df['Year'].isin(TEST_YEARS)].copy()
df_test_aub  = df_test_all[df_test_all['institution'] == 'AUB'].copy()
df_aub_train = df_train[df_train['institution'] == 'AUB'].copy()

thr = df_aub_train['Citations'].quantile(QUANTILE)

y_train_merged = (df_train['Citations']     >= thr).astype(int)
y_test_aub     = (df_test_aub['Citations']  >= thr).astype(int)

print(f"Merged train: {len(df_train):,}  |  AUB test: {len(df_test_aub):,}")
print(f"Threshold: {thr:.0f} citations  |  Positive rate (train): {y_train_merged.mean():.1%}")

## 2. Feature engineering

In [ ]:
COL_MAP = {
    'snip':             'SNIP (publication year)',
    'snip_pct':         'SNIP percentile',
    'citescore':        'CiteScore (publication year)',
    'citescore_pct':    'CiteScore percentile',
    'sjr':              'SJR (publication year)',
    'sjr_pct':          'SJR percentile',
    'topic_prom':       'Topic Prominence Percentile',
    'num_authors':      'Authors',
    'num_institutions': 'Affiliations',
    'num_countries':    'Countries',
}


def extract_raw_numeric(subset_df):
    """Extract the base 10 numeric features."""
    vf = pd.DataFrame(index=subset_df.index)
    for feat, col in COL_MAP.items():
        if col in subset_df.columns:
            if col in ('Authors', 'Affiliations', 'Countries'):
                vf[feat] = subset_df[col].fillna('').apply(
                    lambda x: len(str(x).split(';')) if x else 1
                )
            else:
                vf[feat] = pd.to_numeric(subset_df[col], errors='coerce')
    return vf


def add_interactions(vf):
    """Add interaction and ratio features to a raw numeric DataFrame.

    Each interaction is only computed when all required source columns are
    present — gracefully skips features whose inputs are missing from the data.
    """
    v = vf.copy()
    cols = set(v.columns)

    # Venue quality interactions
    if {'citescore_pct', 'snip_pct'} <= cols:
        v['venue_quality']      = v['citescore_pct'] * v['snip_pct'] / 100
    if {'citescore_pct', 'topic_prom'} <= cols:
        v['venue_momentum']     = v['citescore_pct'] * v['topic_prom'] / 100
    if {'sjr_pct', 'snip_pct'} <= cols:
        v['venue_disagreement'] = (v['sjr_pct'] - v['snip_pct']).abs()
    if {'snip', 'topic_prom'} <= cols:
        v['top_venue_topic']    = v['snip'] * v['topic_prom'] / 100

    # Collaboration ratios (clip to avoid div-by-zero)
    if 'num_authors' in cols:
        authors = v['num_authors'].clip(lower=1)
        if 'num_countries' in cols:
            v['collab_breadth'] = v['num_countries'] / authors
            v['multi_country']  = (v['num_countries'] > 1).astype(float)
        if 'num_institutions' in cols:
            v['inst_diversity'] = v['num_institutions'] / authors
            v['author_load']    = authors / v['num_institutions'].clip(lower=1)

    return v


def get_extended_numeric(df_tr, df_te):
    """Extract, engineer, impute, and scale numeric features."""
    raw_tr = extract_raw_numeric(df_tr)
    raw_te = extract_raw_numeric(df_te)

    # Impute with training median before interactions
    tr_median = raw_tr.median()
    raw_tr = raw_tr.fillna(tr_median)
    raw_te = raw_te.fillna(tr_median)

    ext_tr = add_interactions(raw_tr)
    ext_te = add_interactions(raw_te)

    scaler = StandardScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(ext_tr), index=ext_tr.index, columns=ext_tr.columns)
    X_te = pd.DataFrame(scaler.transform(ext_te),     index=ext_te.index, columns=ext_te.columns)
    return X_tr, X_te


# Preview feature set
sample_raw = extract_raw_numeric(df_train.head(3))
sample_ext = add_interactions(sample_raw.fillna(0))
new_feats  = [c for c in sample_ext.columns if c not in sample_raw.columns]
print(f"Raw features:      {len(sample_raw.columns)} → {list(sample_raw.columns)}")
print(f"New interactions:  {len(new_feats)} → {new_feats}")
print(f"Total features:    {len(sample_ext.columns)}")


## 3. Build feature matrices

In [ ]:
# Extended numerics
X_ext_tr, X_ext_te_aub = get_extended_numeric(df_train, df_test_aub)
print(f"Extended numeric: train {X_ext_tr.shape}  test {X_ext_te_aub.shape}")

# Load SPECTER1 embeddings from nb55 cache
SPECTER1_CACHE = CACHE_DIR / 'specter_embeddings_merged.pkl'
specter_cols   = None
X_sp1_tr       = None
X_sp1_te_aub   = None

if SPECTER1_CACHE.exists():
    with open(SPECTER1_CACHE, 'rb') as f:
        cache = pickle.load(f)
    emb_tr  = cache['train_merged']
    emb_te  = cache['test_aub']
    specter_cols = [f'sp_{i}' for i in range(emb_tr.shape[1])]

    sp_scaler = StandardScaler()
    sp_tr_scaled = pd.DataFrame(
        sp_scaler.fit_transform(emb_tr),
        index=df_train.index, columns=specter_cols
    )
    sp_te_scaled = pd.DataFrame(
        sp_scaler.transform(emb_te),
        index=df_test_aub.index, columns=specter_cols
    )

    X_sp1_tr     = pd.concat([sp_tr_scaled, X_ext_tr.set_index(sp_tr_scaled.index)],      axis=1)
    X_sp1_te_aub = pd.concat([sp_te_scaled, X_ext_te_aub.set_index(sp_te_scaled.index)],  axis=1)
    print(f"SPECTER1 + extended numeric: train {X_sp1_tr.shape}  test {X_sp1_te_aub.shape}")
else:
    print("SPECTER1 cache not found — skipping SPECTER1+extended configs. Run nb55 first.")

## 4. Evaluation helper

In [ ]:
n_pos = int(y_train_merged.sum())
n_neg = int((y_train_merged == 0).sum())
scale_w = n_neg / n_pos

PARAM_DIST = {
    'n_estimators':      [200, 300, 500, 700],
    'learning_rate':     [0.01, 0.03, 0.05, 0.1],
    'num_leaves':        [15, 31, 63, 127],
    'min_child_samples': [10, 20, 30, 50],
    'subsample':         [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9],
    'reg_alpha':         [0.0, 0.1, 0.5, 1.0],
    'reg_lambda':        [0.0, 0.1, 0.5, 1.0],
}


def evaluate(X_tr, y_tr, X_te, y_te, label, tune=True):
    """Train LGBM (optionally tuned), grid-search threshold, return metrics."""
    base = LGBMClassifier(scale_pos_weight=scale_w, random_state=RANDOM_STATE, verbose=-1)

    if tune:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        search = RandomizedSearchCV(
            base, PARAM_DIST, n_iter=50, scoring='f1',
            cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=0
        )
        search.fit(X_tr, y_tr)
        model = search.best_estimator_
        cv_f1 = search.best_score_
        best_params = search.best_params_
    else:
        model = LGBMClassifier(
            n_estimators=500, learning_rate=0.05, num_leaves=31,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_w, random_state=RANDOM_STATE, verbose=-1
        )
        model.fit(X_tr, y_tr)
        cv_f1 = None
        best_params = {}

    proba = model.predict_proba(X_te)[:, 1]
    thresholds = np.arange(0.10, 0.91, 0.01)
    f1s    = [f1_score(y_te, (proba >= t).astype(int), zero_division=0) for t in thresholds]
    best_t = thresholds[int(np.argmax(f1s))]
    y_pred = (proba >= best_t).astype(int)

    return {
        'label':       label,
        'f1':          f1_score(y_te, y_pred, zero_division=0),
        'auc':         roc_auc_score(y_te, proba),
        'kappa':       cohen_kappa_score(y_te, y_pred),
        'recall':      recall_score(y_te, y_pred, zero_division=0),
        'precision':   precision_score(y_te, y_pred, zero_division=0),
        'threshold':   best_t,
        'cv_f1':       cv_f1,
        'best_params': best_params,
        'model':       model,
    }


def print_result(r):
    df1  = r['f1']  - NB54_REF_CLEAN_F1
    dauc = r['auc'] - NB54_REF_CLEAN_AUC
    print(f"  F1:    {r['f1']:.4f}  ({df1:+.4f} vs REF-CLEAN)")
    print(f"  AUC:   {r['auc']:.4f}  ({dauc:+.4f} vs REF-CLEAN)")
    print(f"  Kappa: {r['kappa']:.4f}")
    print(f"  Recall: {r['recall']:.4f}  |  Precision: {r['precision']:.4f}  |  Threshold: {r['threshold']:.2f}")
    if r['cv_f1']:
        print(f"  CV F1: {r['cv_f1']:.4f}")


print(f'scale_pos_weight: {scale_w:.2f}')
print('Evaluation helper ready.')

## 5. Experiments

### 5a. Config A — Extended numerics only, LGBM tuned (ablation)

In [ ]:
results = []

print('=' * 65)
print('Config A: Extended numerics only, LGBM tuned, merged→AUB')
print('=' * 65)

res_a = evaluate(
    X_ext_tr.values, y_train_merged,
    X_ext_te_aub.values, y_test_aub,
    label='Config A (ext-numeric only, LGBM tuned, merged→AUB)',
    tune=True,
)
results.append(res_a)
print_result(res_a)

### 5b. Config B — Raw numerics only, LGBM tuned (nb55-E equivalent for fair comparison)

In [ ]:
print('=' * 65)
print('Config B: Raw numerics only, LGBM tuned, merged→AUB (baseline)')
print('=' * 65)

# Raw numeric only — same 10 features as nb55
raw_tr_df, raw_te_df = (lambda r_tr, r_te, med: (
    pd.DataFrame(
        StandardScaler().fit_transform(r_tr.fillna(med)),
        index=r_tr.index, columns=r_tr.columns
    ),
    pd.DataFrame(
        StandardScaler().fit(
            r_tr.fillna(med)
        ).transform(r_te.fillna(med)),
        index=r_te.index, columns=r_te.columns
    ),
))(
    *(lambda a, b: (a, b))(
        extract_raw_numeric(df_train),
        extract_raw_numeric(df_test_aub)
    ),
    extract_raw_numeric(df_train).median()
)

raw_tr_raw = extract_raw_numeric(df_train)
raw_te_raw = extract_raw_numeric(df_test_aub)
med = raw_tr_raw.median()
sc  = StandardScaler()
X_raw_tr  = pd.DataFrame(sc.fit_transform(raw_tr_raw.fillna(med)), index=raw_tr_raw.index, columns=raw_tr_raw.columns)
X_raw_te  = pd.DataFrame(sc.transform(raw_te_raw.fillna(med)),     index=raw_te_raw.index, columns=raw_te_raw.columns)

res_b = evaluate(
    X_raw_tr.values, y_train_merged,
    X_raw_te.values, y_test_aub,
    label='Config B (raw-numeric only, LGBM tuned, merged→AUB)',
    tune=True,
)
results.append(res_b)
print_result(res_b)

### 5c. Config C — SPECTER1 + extended numerics, LGBM tuned (main experiment)

In [ ]:
print('=' * 65)
print('Config C: SPECTER1 + extended numerics, LGBM tuned, merged→AUB')
print('=' * 65)

if X_sp1_tr is None:
    print('Skipped — SPECTER1 cache not found. Run nb55 first.')
    res_c = None
else:
    res_c = evaluate(
        X_sp1_tr.values, y_train_merged,
        X_sp1_te_aub.values, y_test_aub,
        label='Config C (SPECTER1+ext-numeric, LGBM tuned, merged→AUB)',
        tune=True,
    )
    results.append(res_c)
    print_result(res_c)

## 6. Feature importance

In [ ]:
import matplotlib.pyplot as plt

# Extended-numeric-only model (Config A) — interpretable importance
model_a = res_a['model']
feat_names = X_ext_tr.columns.tolist()
importances = model_a.feature_importances_

imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True)

new_feats_set = {'venue_quality', 'venue_momentum', 'venue_disagreement',
                 'top_venue_topic', 'collab_breadth', 'inst_diversity',
                 'author_load', 'multi_country'}
colors = ['#4C8BE2' if f in new_feats_set else '#888888' for f in imp_df['feature']]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp_df['feature'], imp_df['importance'], color=colors)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#4C8BE2', label='New interaction features'),
    Patch(color='#888888', label='Original 10 features'),
], fontsize=9)

ax.set_title('Feature Importance — Config A (Extended Numerics, LGBM tuned)', fontsize=12)
ax.set_xlabel('Importance (split gain)')
plt.tight_layout()

docs_dir = Path('../../docs')
docs_dir.mkdir(exist_ok=True)
plt.savefig(docs_dir / 'nb57_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance plot saved.')

## 7. Results summary

In [ ]:
ref_rows = [
    {'label': 'REF-CLEAN (TF-IDF+num, LR, AUB→AUB)',
     'f1': NB54_REF_CLEAN_F1, 'auc': NB54_REF_CLEAN_AUC, 'kappa': None,
     'recall': None, 'precision': None, 'cv_f1': None},
    {'label': 'nb54-E (SPECTER1+num, LGBM tuned, AUB→AUB)',
     'f1': NB54_BEST_F1, 'auc': NB54_BEST_AUC, 'kappa': None,
     'recall': None, 'precision': None, 'cv_f1': None},
]

if NB55_C_F1:
    ref_rows.append({
        'label': 'nb55-C (SPECTER1+num, LGBM tuned, merged→AUB)',
        'f1': NB55_C_F1, 'auc': NB55_C_AUC, 'kappa': None,
        'recall': None, 'precision': None, 'cv_f1': None,
    })

res_df = pd.DataFrame(ref_rows + [r for r in results if r is not None])
res_df['delta_f1']  = res_df['f1']  - NB54_REF_CLEAN_F1
res_df['delta_auc'] = res_df['auc'] - NB54_REF_CLEAN_AUC

print('\n' + '=' * 120)
print('RESULTS SUMMARY — Notebook 57: Numeric Feature Engineering')
print('=' * 120)
cols = ['label', 'f1', 'delta_f1', 'auc', 'delta_auc', 'kappa']
print(res_df[cols].to_string(index=False, float_format='{:.4f}'.format))

best = res_df.loc[res_df['f1'].idxmax()]
print(f"\nBest config: {best['label']}")
print(f"  F1:  {best['f1']:.4f}  ({best['delta_f1']:+.4f} vs REF-CLEAN)")
print(f"  AUC: {best['auc']:.4f}")
print(f"  Gap to supervisor target (0.75): {0.75 - best['f1']:+.4f}")

# Did extended numerics help vs raw numerics?
print("\n--- Interaction features impact ---")
r_a = next((r for r in results if r and 'ext-numeric only' in r['label']), None)
r_b = next((r for r in results if r and 'raw-numeric only' in r['label']), None)
if r_a and r_b:
    delta = r_a['f1'] - r_b['f1']
    verdict = 'HELPED' if delta > 0.005 else ('HURT' if delta < -0.005 else 'FLAT')
    print(f"Extended vs raw (numeric-only):  ΔF1={delta:+.4f}  → {verdict}")

r_c = next((r for r in results if r and 'SPECTER1+ext' in r['label']), None)
if r_c and NB55_C_F1:
    delta_c = r_c['f1'] - NB55_C_F1
    verdict_c = 'HELPED' if delta_c > 0.005 else ('HURT' if delta_c < -0.005 else 'FLAT')
    print(f"SPECTER1+ext vs SPECTER1+raw:    ΔF1={delta_c:+.4f}  → {verdict_c}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#888888' if l.startswith('REF') or l.startswith('nb5') else
          ('#4C8BE2' if d >= 0 else '#E24C4C')
          for l, d in zip(res_df['label'], res_df['delta_f1'])]
labels_short = [l.split('(')[0].strip() if '(' in l else l for l in res_df['label']]

for ax, col, title, ref_val, ref_label in [
    (axes[0], 'f1',  'F1 Score', NB54_REF_CLEAN_F1, 'REF-CLEAN'),
    (axes[1], 'auc', 'ROC-AUC',  NB54_REF_CLEAN_AUC, 'REF-CLEAN'),
]:
    ax.barh(labels_short, res_df[col], color=colors)
    ax.axvline(ref_val, color='grey', linestyle='--', linewidth=1.5, label=ref_label)
    if col == 'f1':
        ax.axvline(0.75, color='green', linestyle=':', linewidth=1.5, label='Target 0.75')
    ax.set_title(title)
    ax.legend(fontsize=8)
    deltas = res_df['delta_f1'] if col == 'f1' else res_df['delta_auc']
    for i, (v, d) in enumerate(zip(res_df[col], deltas)):
        ax.text(v + 0.001, i, f'{v:.4f} ({d:+.4f})', va='center', fontsize=8)

plt.suptitle('Notebook 57 — Numeric Feature Engineering', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(docs_dir / 'nb57_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Results plot saved.')

## 8. Conclusions

In [ ]:
print('=' * 70)
print('NOTEBOOK 57 — CONCLUSIONS')
print('=' * 70)

for row in ref_rows:
    d = row['f1'] - NB54_REF_CLEAN_F1
    print(f"  REFERENCE  {row['label']:60s}  F1={row['f1']:.4f} ({d:+.4f})  AUC={row['auc']:.4f}")

for r in results:
    if r is None:
        continue
    df1  = r['f1']  - NB54_REF_CLEAN_F1
    dauc = r['auc'] - NB54_REF_CLEAN_AUC
    outcome = 'IMPROVED' if df1 > 0.01 else ('DEGRADED' if df1 < -0.01 else 'FLAT')
    kstr = f"  Kappa={r['kappa']:.4f}" if r['kappa'] is not None else ''
    print(f"  {outcome:9s}  {r['label']:60s}  F1={r['f1']:.4f} ({df1:+.4f})  AUC={r['auc']:.4f} ({dauc:+.4f}){kstr}")

best_r = max((r for r in results if r), key=lambda x: x['f1'])
print(f"\nGap to supervisor target (F1=0.75): {0.75 - best_r['f1']:+.4f}")